In [1]:
# Task 4.1

import csv
import sqlite3

conn = sqlite3.connect("Busroutes.db")

with open("Service.csv") as f:
    reader = csv.reader(f)
    header = next(reader)
    
    for line in reader:
        conn.execute('''
        INSERT INTO Service(ServiceNo, Operator, Category)
        VALUES(?,?,?)
        ''', (line[0], line[1], line[2]))
        

with open("Stop.csv") as f:
    reader = csv.reader(f)
    header = next(reader)
    
    for line in reader:
        conn.execute('''
        INSERT INTO Stop(BusStopCode, RoadName, Description)
        VALUES(?,?,?)
        ''', (line[0], line[1], line[2]))
        
with open("Route.csv") as f:
    reader = csv.reader(f)
    header = next(reader)
    
    for line in reader:
        conn.execute('''
        INSERT INTO Route(ServiceNo, Direction, StopSequence, BusStopCode)
        VALUES(?,?,?,?)
        ''', (line[0], line[1], line[2], line[3]))
        
conn.commit()
conn.close()

IntegrityError: UNIQUE constraint failed: Service.ServiceNo

In [2]:
# Task 4.2

import sqlite3

bus_no = input("Please input a bus service number: ")

conn = sqlite3.connect("Busroutes.db")

cursor = conn.execute('''
SELECT Route.Direction, Route.BusStopCode, Stop.RoadName, Stop.Description
FROM Route, Stop
WHERE Route.BusStopCode = Stop.BusStopCode
AND Route.ServiceNo = ?
ORDER BY Route.Direction, Route.StopSequence
''', (bus_no, ))

for line in cursor.fetchall():
    print(line)
    
conn.close()

Please input a bus service number: 


In [3]:
# Task 4.3
import sqlite3
import flask
from flask import request, render_template, url_for

app = flask.Flask(__name__)

@app.route('/')
def home():
    return render_template('home.html')

@app.route('/display/')
def display():
    bus_stop = request.args.get('busstop')
    print(bus_stop)
    
    conn = sqlite3.connect("Busroutes.db")
    
    cursor = conn.execute(
    '''SELECT Route.ServiceNo, Service.Operator
    FROM Route, Service
    WHERE Route.ServiceNo = Service.ServiceNo
    AND Route.BusStopCode = ?''', (bus_stop,))

    results = cursor.fetchall()
    
    if len(results) == 0:
        valid = False
    else:
        valid = True
    
    conn.close()
        
    return render_template('display.html', bus_stop = bus_stop, results = results, valid = valid)

if __name__ == '__main__':
    app.run()


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [20/Aug/2025 11:39:55] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [20/Aug/2025 11:39:55] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [20/Aug/2025 11:40:01] "GET /display/?busstop=01112 HTTP/1.1" 200 -


01112


127.0.0.1 - - [20/Aug/2025 11:40:14] "GET /display/?busstop=01111 HTTP/1.1" 200 -


01111
